# 10 · F.R.I.D.A.Y. – prove it works: datasets and evaluations

*"Does it work?"* is not answered by trying three prompts. An **evaluation** runs a fixed **dataset** of test inputs through your model or agent and scores the answers with **evaluators**. Do it before every release and you get a **quality gate** instead of a feeling.

| Concept | Meaning |
|---|---|
| **Dataset** | versioned JSONL/CSV in your project (`query`, optionally `response`, `ground_truth`) |
| **Evaluator** | a scorer: AI-judged (`builtin.task_adherence`, `builtin.coherence`, ...), safety (`builtin.violence`), or classic metrics (`builtin.f1_score`) |
| **Evaluation** (`evals.create`) | the *definition*: data schema + evaluators |
| **Run** (`evals.runs.create`) | one execution of that definition against a data source or target |
| **Result** | per item: `score`, `label`/`passed`, `reason`; per run: pass rate per evaluator |

We will evaluate a prompt agent **twice** (vague vs strict instructions), compare, and gate on the result. Deep dive: [docs/datasets-evals.md](../docs/datasets-evals.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_evals import show_scores, show_pass_rates

from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool

project_openai = openai_client()   # the project's OpenAI client: vector stores, evals, runs
to_clean = {"agents": [], "evals": [], "vector_stores": [], "datasets": []}

## 1 · Upload a dataset
`datasets.upload_file` creates a **named, versioned** dataset. Re-upload with a new `version` when the test set changes; runs always reference an exact version, so results stay comparable.

In [ ]:
QUERIES_FILE = ROOT / "data" / "eval-queries.jsonl"
print(QUERIES_FILE.read_text())
query_dataset = project().datasets.upload_file(name="stark-eval-queries", version="1", file_path=str(QUERIES_FILE))
to_clean["datasets"].append(query_dataset.name)
print("dataset id:", query_dataset.id)

## 2 · The system under test
A prompt agent that answers from the [workshop handbook](../data/workshop-handbook.md) (file search). Version 1 has **vague** instructions on purpose: `VAGUE_INSTRUCTIONS` is one sentence that says nothing about the handbook. Section 6 changes only the instructions.

In [ ]:
handbook_store = project_openai.vector_stores.create(name="stark-handbook-nb10")
to_clean["vector_stores"].append(handbook_store.id)
with open(ROOT / "data" / "workshop-handbook.md", "rb") as f:
    project_openai.vector_stores.files.upload_and_poll(vector_store_id=handbook_store.id, file=f)

AGENT_NAME = "jarvis-nb10"
VAGUE_INSTRUCTIONS = "You are a helpful assistant."
vague_agent = project().agents.create_version(agent_name=AGENT_NAME, definition=PromptAgentDefinition(
    model=model("chat"), instructions=VAGUE_INSTRUCTIONS, tools=[FileSearchTool(vector_store_ids=[handbook_store.id])]))
to_clean["agents"].append(AGENT_NAME)
print("agent", vague_agent.name, "version", vague_agent.version)

## 3 · Define the evaluation
`item_schema` describes one dataset row. Each evaluator needs a `data_mapping`: `{{item.query}}` is a dataset field, `{{sample.output_text}}` is the agent's text answer, `{{sample.output_items}}` is the full interaction including tool calls (agent evaluators need it).

Two helpers from [stark_helpers.py](stark_helpers.py) keep the cell short: `item_schema(...)` describes a row, `criterion(name, evaluator, mapping)` picks a built-in evaluator and lets the judge model (`STARK_JUDGE_MODEL`) score it.

In [ ]:
data_source_config = item_schema({"query": {"type": "string"}}, required=["query"])   # sample=True: the agent's answer is {{sample.*}}

answer_text = {"query": "{{item.query}}", "response": "{{sample.output_text}}"}
answer_items = {"query": "{{item.query}}", "response": "{{sample.output_items}}"}   # text plus tool calls
testing_criteria = [
    criterion("coherence", "coherence", answer_text),
    criterion("task_adherence", "task_adherence", answer_items),
    criterion("intent_resolution", "intent_resolution", answer_items),
    criterion("violence", "violence", answer_text, judge=False),   # a safety evaluator: no judge model needed
]
quality_evaluation = project_openai.evals.create(name="stark-agent-quality", data_source_config=data_source_config,
                                                 testing_criteria=testing_criteria)
to_clean["evals"].append(quality_evaluation.id)
print("evaluation:", quality_evaluation.id)

## 4 · Run it against agent version 1
The `azure_ai_target_completions` data source sends every dataset row to the **target** (our agent), collects the answers, and scores them. `agent_target(query_dataset.id, vague_agent.name, vague_agent.version)` builds it:

```json
{"type": "azure_ai_target_completions",
 "source": {"type": "file_id", "id": "<dataset id>"},
 "input_messages": {"type": "template", "template": [
     {"type": "message", "role": "user", "content": {"type": "input_text", "text": "{{item.query}}"}}]},
 "target": {"type": "azure_ai_agent", "name": "jarvis-nb10", "version": "1"}}
```

In [ ]:
vague_run = project_openai.evals.runs.create(eval_id=quality_evaluation.id, name="vague-instructions",
                                             data_source=agent_target(query_dataset.id, vague_agent.name, vague_agent.version))
print("started", vague_run.id)
vague_run = wait_eval_run(quality_evaluation.id, vague_run.id, project_openai)   # polls until the run ends
print("pass rates:", pass_rates(vague_run))
print("portal report:", getattr(vague_run, "report_url", None))

## 5 · Read the results
Every row, every evaluator: a pass or fail, a score and a **reason** (the judge explains itself). `show_scores()` ([stark_evals.py](stark_evals.py)) reads them with `evals.runs.output_items.list(...)` and shows one table: a row per query, a column per evaluator. Below it come the agent's answers (folded, click to open) and the full reasons of the failed results. It looks like this (one row; the run may list the queries in any order):

| # | input | coherence | task_adherence | intent_resolution | violence |
|---|---|---|---|---|---|
| 1 | What is the workshop Wi-Fi password? | ✅ or ❌, then the score | ... | ... | ... |

**Expected:** ❓ the scores come from a judge model, so they differ between runs. `violence` passes on every row: no query asks for anything violent. The rows to read first are the ones the handbook does not answer: the Wi-Fi password and the limerick about Dummy. With `VAGUE_INSTRUCTIONS` nothing stops the agent from guessing a password or writing the limerick.

In [ ]:
show_scores(quality_evaluation.id, vague_run.id, project_openai)

## 6 · Fix, re-run, compare
Two steps: a new agent version with strict instructions, run against the same evaluation; then the two runs side by side.

### Step 1 · Strict instructions

**Goal:** change one thing, the instructions, and measure the effect. Same agent name (it becomes version 2), same handbook, same dataset, same evaluators.

| Input | Version 1 | Version 2 |
|---|---|---|
| instructions | `VAGUE_INSTRUCTIONS`: "You are a helpful assistant." | `STRICT_INSTRUCTIONS`: answer only from the handbook, say "That is not in the handbook." instead of guessing, refuse unrelated requests, two sentences max |
| run name | `vague-instructions` | `strict-instructions` |

Run sections 2 to 4 first: this cell needs `handbook_store`, `quality_evaluation` and `query_dataset`.

**Expected:** ❓ judge scores differ between runs. On the Wi-Fi password the strict agent answers "That is not in the handbook." It refuses the limerick; `intent_resolution` may score that refusal lower than the vague agent's limerick. That is a real trade-off, and the judge's reasons say so.

In [ ]:
STRICT_INSTRUCTIONS = (
    "You are J.A.R.V.I.S., the workshop assistant. Answer ONLY from the Stark Workshop Handbook (use file search). "
    "If the handbook does not contain the answer, say 'That is not in the handbook.' and do not guess. "
    "Refuse requests unrelated to the workshop politely. Maximum two sentences."
)
strict_agent = project().agents.create_version(agent_name=AGENT_NAME, definition=PromptAgentDefinition(
    model=model("chat"), instructions=STRICT_INSTRUCTIONS, tools=[FileSearchTool(vector_store_ids=[handbook_store.id])]))
print("agent", strict_agent.name, "version", strict_agent.version)

strict_run = project_openai.evals.runs.create(eval_id=quality_evaluation.id, name="strict-instructions",
                                              data_source=agent_target(query_dataset.id, strict_agent.name, strict_agent.version))
strict_run = wait_eval_run(quality_evaluation.id, strict_run.id, project_openai)
show_scores(quality_evaluation.id, strict_run.id, project_openai)

### Step 2 · Compare the two runs

**Goal:** one table to decide on: the pass rate per evaluator, version 1 next to version 2. `show_pass_rates()` ([stark_evals.py](stark_evals.py)) builds it from `pass_rates(run)`.

**Expected:** ❓ `violence` stays at 100% in both columns. Whether `task_adherence` and `intent_resolution` go up depends on the judge: read the reasons in step 1 before you trust a percentage.

In [ ]:
show_pass_rates({"v1 vague": vague_run, "v2 strict": strict_run})

## 7 · A quality gate
Wrap the check in one line and run it in CI after `deploy_agents.py`: the pipeline fails when quality drops. (`assert_pass_rate` lives in [stark_helpers.py](stark_helpers.py).)

**Goal:** the gate a pipeline would run on version 2. With 8 queries and `minimum=0.75`, every evaluator must pass at least 6 rows.

**Expected:** the pass rates, then either `quality gate passed (>= 0.75)` or `GATE FAILED:` with the evaluators below 75%. ❓ Which one depends on the judge's scores in section 6. The `try` keeps the notebook running when the gate fails; in CI you let the error fail the job.

In [ ]:
try:
    assert_pass_rate(strict_run, minimum=0.75)
except AssertionError as exc:
    print("GATE FAILED:", exc)

## 8 · Evaluate saved responses (no agent call)
You already have answers (logged, from another system, from a human)? Use the `jsonl` data source: rows carry `query`, `response` and `ground_truth`, and classic metrics such as F1 compare them. First the evaluation: `sample=False`, because no target answers.

In [ ]:
saved_dataset = project().datasets.upload_file(name="stark-eval-saved", version="1", file_path=str(ROOT / "data" / "eval-with-responses.jsonl"))
to_clean["datasets"].append(saved_dataset.name)
saved_schema = item_schema({"query": {"type": "string"}, "response": {"type": "string"}, "ground_truth": {"type": "string"}},
                           required=["query", "response"], sample=False)   # no target: the answers are already in the rows
saved_evaluation = project_openai.evals.create(name="stark-saved-responses", data_source_config=saved_schema, testing_criteria=[
    criterion("f1", "f1_score", {"response": "{{item.response}}", "ground_truth": "{{item.ground_truth}}"}, judge=False),
    criterion("coherence", "coherence", {"query": "{{item.query}}", "response": "{{item.response}}"}),
])
to_clean["evals"].append(saved_evaluation.id)
print("evaluation:", saved_evaluation.id)

### Step · Fluent is not the same as right

**Goal:** see two evaluators disagree. F1 counts the words a response shares with the ground truth; coherence asks a judge whether the text reads well. The rows come from [eval-with-responses.jsonl](../data/eval-with-responses.jsonl):

| # | Saved response | Ground truth |
|---|---|---|
| 1 | Pepper Potts approves all purchases above 5,000 USD. | Pepper Potts approves purchases above 5,000 USD. |
| 2 | Yes, it is fine to service it by yourself. | No. The arc reactor must never be serviced alone. |
| 3 | Happy Hogan drives. | Happy Hogan drives. |
| 4 | The doors close after about half a minute. | The fabrication bay doors close automatically after 30 seconds. |

**Expected:** (row numbers as in the file; the run's table may list them in another order) F1 is computed, not judged, so it is the same in every run. `builtin.f1_score` passes at 0.5. Row 1 passes (about 0.9), row 3 scores 1.0. Row 2 shares no words with the ground truth: F1 0, fail. Row 4 fails too (about 0.4), although it is right: half a minute is 30 seconds, but F1 counts words, not meaning. Coherence: ❓ a judge score, but all four answers are fluent, so expect them to pass, row 2 included. Fluent and wrong: that is why you combine evaluators.

In [ ]:
saved_run = project_openai.evals.runs.create(eval_id=saved_evaluation.id, name="saved",
                                             data_source={"type": "jsonl", "source": {"type": "file_id", "id": saved_dataset.id}})
saved_run = wait_eval_run(saved_evaluation.id, saved_run.id, project_openai)
show_scores(saved_evaluation.id, saved_run.id, project_openai, reasons="all")

## 9 · More ways to get a dataset
Not used in this notebook, but worth knowing:

- **Synthetic**: let Foundry generate test queries for a prelaunch agent – see [Generate a synthetic evaluation dataset](https://learn.microsoft.com/azure/foundry/observability/how-to/evaluation-dataset-synthetic).
- **From traces**: turn real production conversations into a dataset – see [Convert agent traces into evaluation datasets](https://learn.microsoft.com/azure/foundry/observability/how-to/traces-to-dataset).
- **Custom evaluators**: prompt-based or code-based scorers for your own rules.

## 👀 In the portal
**Evaluation**: the evaluations `stark-agent-quality` and `stark-saved-responses`. Open a run for the row scores. In `stark-agent-quality`, select the runs `vague-instructions` and `strict-instructions` and use **Compare** (v1 against v2). Look before you run 🧹 Clean up: it deletes both evaluations.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
for eid in to_clean["evals"]:
    try: project_openai.evals.delete(eid)
    except Exception as exc: print("skip eval", eid, type(exc).__name__)
for vid in to_clean["vector_stores"]:
    project_openai.vector_stores.delete(vid)
for name in to_clean["agents"]:
    delete_agent(name)
for name in to_clean["datasets"]:
    delete_dataset(name)

## 🎯 Recap
| Step | Call |
|---|---|
| Data | `datasets.upload_file(name, version, file_path)` |
| Definition | `evals.create(data_source_config, testing_criteria)` |
| Run | `evals.runs.create(eval_id, data_source={... target ...})` |
| Results | `evals.runs.output_items.list(...)`, `pass_rates(run)` |

> 💡 Next: [11 · Conversation and trace evaluations](11_conversation_and_trace_evals.ipynb) – judge whole conversations and live traffic.